In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Tabel verb + ma andmete kogumine


**Ülesande püstitus**
    
Kõik laused, kus esineb tabelis [list_ma.csv](list_ma.csv) olev verb ja verbil on otsene alluv deprel=xcomp, feats sisaldab sup

**Tulemus** 

Tabel veergudega:
1. leitud lause, 
2. milline tabelis olevates verbidest seal esineb (algvormis), 
3. ma-infinitiivi vormis oleva verbi lemma, 
4. keeletase.
   

In [8]:
import pandas as pd
from datetime import datetime
from notebook_context import corpus_reader, LISTS_FOLDER

date_time = datetime.now().strftime("%Y%m%d-%H%M%S")



MA_VERBS_LIST =   "./lists/102.list_ma.csv"
RESULTS_FILE= LISTS_FOLDER / f"results/verb_xcompSup_{date_time}.csv"

In [9]:
%%time

# verbid etteantud nimekirjast
df_verbs = pd.read_csv(MA_VERBS_LIST)
my_verbs = list(df_verbs['lemma'].unique())


CPU times: user 0 ns, sys: 2.9 ms, total: 2.9 ms
Wall time: 2.67 ms


In [10]:
my_verbs

['agiteerima',
 'ahvatlema',
 'aitama',
 'ajama',
 'ajendama',
 'allutama  ',
 'ambuma',
 'arvama',
 'asetama',
 'asustama',
 'avatlema',
 'ehmatama',
 'ehtima',
 'ergutama',
 'harjama',
 'harjutama',
 'hurjutama',
 'hõikama',
 'hõõruma',
 'häälestama',
 'igatsema',
 'ihalema',
 'innustama',
 'inspireerima',
 'intrigeerima',
 'istutama',
 'juhtima',
 'julgustama',
 'jätma',
 'kaasama',
 'kallutama',
 'kamandama',
 'kangutama',
 'kannustama',
 'kasvatama',
 'keelitama',
 'kehutama',
 'kiskuma',
 'kitsendama ',
 'klõpsama',
 'klõpsatama',
 'kohustama',
 'koolitama',
 'kupatama',
 'kutsuma',
 'käivitama',
 'käratama',
 'käsutama',
 'kütma',
 'laskma',
 'laulma',
 'lihvima',
 'lohistama',
 'looma',
 'lubama',
 'lõõtsuma',
 'läkitama',
 'lööma',
 'lükkama',
 'lülitama',
 'mahitama',
 'mahutama ',
 'majutama',
 'manitsema',
 'manööverdama',
 'matma',
 'meelestama',
 'meelitama',
 'mobiliseerima',
 'motiveerima',
 'mõjutama',
 'mõtlema',
 'määrama',
 'naelutama',
 'nihutama',
 'nõudma',
 'nüh

In [11]:
%%time

collected_data = []
count = 0
for collection_id, graph in corpus_reader.get_sentences():
    # matrix for node distances
    dpath = graph.get_distances_matrix()
    
    # verb nodes
    verb_nodes = [v for v in graph.get_nodes_by_attributes(attrname="POS", attrvalue="VERB") if graph.nodes[v]["lemma"] in my_verbs]
    if not len(verb_nodes): continue
    
    # xcomp
    xcomp_nodes = graph.get_nodes_by_attributes(attrname="deprel", attrvalue="xcomp")
   
    if not len(xcomp_nodes): continue
    
    for verb in verb_nodes:
        # childnodes
        kids = [k for k in dpath[verb] if dpath[verb][k] == 1]
        for xcomp in xcomp_nodes:
            if xcomp not in kids:
                continue
            if not graph.nodes[xcomp]["feats"] or "VerbForm" not in graph.nodes[xcomp]["feats"].keys() or not graph.nodes[xcomp]["feats"]["VerbForm"] == 'Sup':
                continue
            
            #graph.draw_graph2(highlight=[verb, xcomp])
            d = {
                'id':  graph.get_metadata('sent_id'),
                'sentence':  graph.get_metadata('text'),
                'verb':  graph.nodes[verb]["lemma"],
                'xcomp':  graph.nodes[xcomp]["lemma"],
                'form':  graph.nodes[xcomp]["form"],
                'sub': " ".join(
                            [graph.nodes[n]["form"] for n in sorted([verb] + kids)]
                        ),
                'keeletase': graph.get_metadata("doc").get("keeletase"),
                'emakeel': graph.get_metadata("doc").get("emakeel"),
                'klass': graph.get_metadata("doc").get("klass")
            }
            
            collected_data.append(d)


../data/vrt-with-meta-corpus-02-06-25_ordered.vrt
CPU times: user 12.2 s, sys: 83.8 ms, total: 12.3 s
Wall time: 12.3 s


In [12]:
df = pd.DataFrame.from_dict(collected_data)
df.to_csv(RESULTS_FILE, index=None)
df.head()

,id,sentence,verb,xcomp,form,sub,keeletase,emakeel,klass
0,4255_5,"Minu koolis õpitakse targaks, aga kollide kool...",õppima,karjuma,karjuma,", aga koolis õpitakse karjuma",0,1,3
1,4485_3,Tahan sind kutsuma.,tahtma,kutsuma,kutsuma,Tahan kutsuma .,5,0,6
2,4524_1,"Tere, Ma tahan kutsuda sind oma sünnipäevale k...",kutsuma,mängima,mängima,kutsuda sind mängima,5,0,6
3,4659_2,Tahan kutsuma sind omale sünnipäevale .Minu sü...,tahtma,kutsuma,kutsuma,Tahan kutsuma toimub mängima !,5,0,6
4,4865_5,"Me peol teeme mängima, jalutama, sööma ja tans...",tegema,mängima,mängima,Me peol teeme mängima .,5,0,6
